# 📖 Notebook 4: Disaster Recovery Drill

## Why This Matters

Netflix runs **Chaos Monkey** — a tool that randomly kills production servers
to ensure their systems can handle failure. Banks run quarterly DR drills.
Microsoft Azure tests failover across entire regions.

The only way to know if your BCDR plan works is to **test it**.

In this notebook, you will simulate a disaster and execute a complete
recovery procedure while measuring your actual RPO and RTO.

## Learning Objectives

- Plan and execute a disaster recovery drill
- Simulate primary database failure
- Execute the full failover procedure
- Measure actual RPO (data lost) and RTO (downtime)
- Generate a DR drill report

## 🛠️ Setup

**Important**: Start from a clean state for accurate measurements!

```bash
cd enterprise-patterns/bcdr
docker-compose down -v && docker-compose up -d
# Wait ~15 seconds for standby to sync
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).

In [ ]:
import psycopg2
import subprocess
import time
import datetime
import json
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}
DB_STANDBY = {
    "host": "localhost", "port": 5433,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def get_standby_connection():
    return psycopg2.connect(**DB_STANDBY)

def docker_exec(container, cmd):
    result = subprocess.run(
        ["docker", "exec", container] + cmd,
        capture_output=True, text=True, timeout=30
    )
    return result.stdout.strip(), result.stderr.strip()

def check_container(name):
    """Check if a Docker container is running."""
    result = subprocess.run(
        ["docker", "inspect", "-f", "{{.State.Running}}", name],
        capture_output=True, text=True
    )
    return result.stdout.strip() == 'true'

# Verify clean state
for name, cfg in [('Primary', DB_PRIMARY), ('Standby', DB_STANDBY)]:
    try:
        conn = psycopg2.connect(**cfg)
        conn.close()
        print(f"✅ {name} is running")
    except Exception as e:
        print(f"❌ {name} is down: {e}")
        print("   Run: docker-compose down -v && docker-compose up -d")

## 📚 DR Drill Plan

A proper DR drill has these phases:

```
Phase 1: PRE-DRILL HEALTH CHECK
  - Verify all services are running
  - Record baseline data counts
  - Verify replication is healthy

Phase 2: SIMULATE DISASTER
  - Write some last-moment data (to measure RPO)
  - Kill the primary database (simulate crash)
  - Record the exact time of failure

Phase 3: DETECT AND RESPOND
  - Detect that the primary is down
  - Verify standby is still running
  - Check standby replication status

Phase 4: EXECUTE FAILOVER
  - Promote standby to primary
  - Verify new primary accepts writes
  - Record recovery time

Phase 5: POST-DRILL ANALYSIS
  - Calculate actual RPO (data lost)
  - Calculate actual RTO (downtime)
  - Generate drill report
```

In [ ]:
# =============================================================================
# Phase 1: PRE-DRILL HEALTH CHECK
# =============================================================================

drill_log = {}  # We will record everything here
drill_log["drill_start"] = datetime.datetime.now().isoformat()

print("=" * 65)
print("PHASE 1: PRE-DRILL HEALTH CHECK")
print("=" * 65)

# Check replication status
primary_conn = get_primary_connection()
primary_cur = primary_conn.cursor()

primary_cur.execute(
    "SELECT client_addr, state, sync_state FROM pg_stat_replication"
)
repl = primary_cur.fetchall()

if repl:
    print(f"\n✅ Replication active: {repl[0][1]} ({repl[0][2]} mode)")
else:
    print("\n❌ No replication! Abort drill.")

# Record baseline data
tables = ['customers', 'orders', 'order_items', 'payments']
baseline = {}
for tbl in tables:
    primary_cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    baseline[tbl] = primary_cur.fetchone()[0]

drill_log["baseline_counts"] = baseline

print("\nBaseline data counts:")
for tbl, count in baseline.items():
    print(f"  {tbl}: {count} rows")

# Record primary WAL position
primary_cur.execute("SELECT pg_current_wal_lsn()")
primary_lsn = str(primary_cur.fetchone()[0])
drill_log["primary_lsn_before_disaster"] = primary_lsn
print(f"\nPrimary WAL position: {primary_lsn}")
print("\n✅ Phase 1 complete. System is healthy.")

primary_conn.close()

In [ ]:
# =============================================================================
# Phase 2: SIMULATE DISASTER
# =============================================================================

print("=" * 65)
print("PHASE 2: SIMULATE DISASTER")
print("=" * 65)

# Write some last-moment data (to test if it survives)
primary_conn = get_primary_connection()
primary_conn.autocommit = True
primary_cur = primary_conn.cursor()

print("\nWriting last-moment data to primary...")
last_moment_ids = []
for i in range(5):
    primary_cur.execute(
        "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
        "VALUES (%s, %s, %s, %s) RETURNING id",
        ('dr_drill', i, 'INSERT', f'last_moment_{i}')
    )
    last_moment_ids.append(primary_cur.fetchone()[0])
    time.sleep(0.1)  # small delay between writes

print(f"  Wrote {len(last_moment_ids)} records (IDs: {last_moment_ids})")
drill_log["last_moment_record_ids"] = last_moment_ids

# Brief pause to allow replication
time.sleep(1)
primary_conn.close()

# === DISASTER STRIKES ===
print("\n💥 SIMULATING PRIMARY FAILURE...")
print("   (Killing primary PostgreSQL container)\n")

disaster_time = time.time()
drill_log["disaster_time"] = datetime.datetime.now().isoformat()

subprocess.run(
    ["docker", "kill", "bcdr-pg-primary"],
    capture_output=True, timeout=10
)

# Verify primary is dead
try:
    test_conn = psycopg2.connect(**DB_PRIMARY)
    test_conn.close()
    print("⚠️  Primary still responding!")
except Exception:
    print("✅ Primary is DOWN. Disaster simulation complete.")
    print(f"   Time of failure: {drill_log['disaster_time']}")

In [ ]:
# =============================================================================
# Phase 3: DETECT AND RESPOND
# =============================================================================

print("=" * 65)
print("PHASE 3: DETECT AND RESPOND")
print("=" * 65)

detection_start = time.time()

# In production, a monitoring system would detect this automatically.
# Here we simulate detection by checking if primary responds.

print("\nDetecting failure...")
try:
    test_conn = psycopg2.connect(**DB_PRIMARY)
    test_conn.close()
    print("  Primary is still up (false alarm)")
except Exception:
    detection_time = time.time() - detection_start
    print(f"  ❌ Primary is unreachable (detected in {detection_time:.2f}s)")
    drill_log["detection_time_seconds"] = detection_time

# Check standby health
print("\nChecking standby health...")
try:
    standby_conn = get_standby_connection()
    standby_cur = standby_conn.cursor()
    standby_cur.execute("SELECT pg_is_in_recovery()")
    is_recovery = standby_cur.fetchone()[0]
    print(f"  ✅ Standby is running (recovery mode: {is_recovery})")

    # Check what data the standby has
    standby_cur.execute(
        "SELECT pg_last_wal_receive_lsn(), pg_last_wal_replay_lsn()"
    )
    recv_lsn, replay_lsn = standby_cur.fetchone()
    print(f"  Last WAL received:  {recv_lsn}")
    print(f"  Last WAL replayed:  {replay_lsn}")

    # Check if last-moment records survived
    print("\n  Checking last-moment records on standby...")
    survived = 0
    for rec_id in last_moment_ids:
        standby_cur.execute(
            "SELECT id FROM audit_log WHERE id = %s", (rec_id,)
        )
        if standby_cur.fetchone():
            survived += 1
    print(f"  Records survived: {survived}/{len(last_moment_ids)}")
    drill_log["records_survived"] = survived
    drill_log["records_total"] = len(last_moment_ids)

    standby_conn.close()
except Exception as e:
    print(f"  ❌ Standby is also down: {e}")
    print("  This would require backup-based recovery (much longer RTO)")

In [ ]:
# =============================================================================
# Phase 4: EXECUTE FAILOVER
# =============================================================================

print("=" * 65)
print("PHASE 4: EXECUTE FAILOVER")
print("=" * 65)

failover_start = time.time()

# Promote standby to primary
print("\nPromoting standby to primary...")
out, err = docker_exec(
    "bcdr-pg-standby",
    ["pg_ctl", "promote", "-D", "/var/lib/postgresql/data"]
)
print(f"  pg_ctl output: {out or err}")

# Wait for promotion
time.sleep(3)

# Verify promotion
print("\nVerifying new primary...")
new_primary_conn = psycopg2.connect(**DB_STANDBY)
new_cur = new_primary_conn.cursor()

new_cur.execute("SELECT pg_is_in_recovery()")
still_recovery = new_cur.fetchone()[0]

if not still_recovery:
    print("  ✅ Standby promoted — no longer in recovery mode!")

    # Test write capability
    new_primary_conn.autocommit = True
    new_cur.execute(
        "INSERT INTO audit_log (table_name, record_id, action, changed_by) "
        "VALUES (%s, %s, %s, %s)",
        ('dr_drill', 999, 'FAILOVER_TEST', 'new_primary_write_test')
    )
    print("  ✅ Write to new primary succeeded!")

    recovery_time = time.time() - disaster_time
    failover_only_time = time.time() - failover_start

    drill_log["recovery_time_seconds"] = recovery_time
    drill_log["failover_time_seconds"] = failover_only_time

    print(f"\n  ⏱️  Total RTO (disaster to recovery): {recovery_time:.1f} seconds")
    print(f"  ⏱️  Failover only: {failover_only_time:.1f} seconds")
else:
    print("  ❌ Server still in recovery mode — promotion may have failed")

new_primary_conn.close()

In [ ]:
# =============================================================================
# Phase 5: POST-DRILL ANALYSIS
# =============================================================================

print("=" * 65)
print("PHASE 5: POST-DRILL ANALYSIS — DR DRILL REPORT")
print("=" * 65)

# Verify data integrity on new primary
new_conn = psycopg2.connect(**DB_STANDBY)
new_cur = new_conn.cursor()

post_counts = {}
for tbl in ['customers', 'orders', 'order_items', 'payments']:
    new_cur.execute(f"SELECT COUNT(*) FROM {tbl}")
    post_counts[tbl] = new_cur.fetchone()[0]

new_conn.close()

# Calculate RPO
records_lost = drill_log["records_total"] - drill_log["records_survived"]

# Print report
print("\n" + "=" * 65)
print("        DISASTER RECOVERY DRILL REPORT")
print("=" * 65)
print(f"\n  Drill Start:    {drill_log['drill_start']}")
print(f"  Disaster Time:  {drill_log['disaster_time']}")

print("\n--- Recovery Metrics ---")
print(f"  Detection Time:    {drill_log.get('detection_time_seconds', 0):.2f} seconds")
print(f"  Failover Time:     {drill_log.get('failover_time_seconds', 0):.1f} seconds")
print(f"  Total RTO:         {drill_log.get('recovery_time_seconds', 0):.1f} seconds")

print("\n--- Data Loss (RPO) ---")
print(f"  Last-moment records written:    {drill_log['records_total']}")
print(f"  Records survived on standby:    {drill_log['records_survived']}")
print(f"  Records LOST:                   {records_lost}")

print("\n--- Data Integrity ---")
table_data = []
for tbl in ['customers', 'orders', 'order_items', 'payments']:
    orig = baseline[tbl]
    post = post_counts[tbl]
    match = "✅" if orig == post else "⚠️"
    table_data.append([tbl, orig, post, match])

print(tabulate(table_data,
    headers=["Table", "Before Disaster", "After Recovery", "Match"],
    tablefmt="grid"))

print("\n--- Verdict ---")
rto = drill_log.get('recovery_time_seconds', 999)
if rto < 30:
    print("  ✅ RTO < 30 seconds — EXCELLENT")
elif rto < 120:
    print("  ✅ RTO < 2 minutes — GOOD")
elif rto < 300:
    print("  ⚠️  RTO < 5 minutes — ACCEPTABLE for non-critical systems")
else:
    print("  ❌ RTO > 5 minutes — NEEDS IMPROVEMENT")

if records_lost == 0:
    print("  ✅ Zero data loss — RPO target met")
else:
    print(f"  ⚠️  {records_lost} records lost — consider synchronous replication")

print("\n" + "=" * 65)
print("  To restore original setup:")
print("  docker-compose down -v && docker-compose up -d")
print("=" * 65)

## 📚 Redis Failover

Our lab also includes Redis replication. Let us check its status.
In production, you would use **Redis Sentinel** or **Redis Cluster** for automatic failover.

In [ ]:
# =============================================================================
# Demo: Check Redis Replication Status
# =============================================================================

import redis

r_primary = redis.Redis(host='localhost', port=6379, decode_responses=True)
r_replica = redis.Redis(host='localhost', port=6380, decode_responses=True)

# Write to primary
r_primary.set("bcdr:test:key", "hello from primary")
time.sleep(0.5)

# Read from replica
replica_val = r_replica.get("bcdr:test:key")

print("=" * 65)
print("REDIS REPLICATION STATUS")
print("=" * 65)

# Primary info
info = r_primary.info('replication')
print(f"\nPrimary role:          {info['role']}")
print(f"Connected replicas:    {info['connected_slaves']}")

# Replica info
rinfo = r_replica.info('replication')
print(f"\nReplica role:          {rinfo['role']}")
print(f"Master host:           {rinfo.get('master_host', 'N/A')}")
print(f"Master link status:    {rinfo.get('master_link_status', 'N/A')}")

print(f"\nReplication test:")
print(f"  Wrote to primary:    'hello from primary'")
print(f"  Read from replica:   '{replica_val}'")
if replica_val == 'hello from primary':
    print("  ✅ Redis replication is working!")

# Cleanup
r_primary.delete("bcdr:test:key")

## 📝 Summary

### What You Accomplished in This Drill

1. **Pre-drill check** — Verified replication health and recorded baseline data.
2. **Simulated disaster** — Killed the primary database container.
3. **Detected failure** — Confirmed primary was unreachable.
4. **Executed failover** — Promoted standby to primary, verified writes.
5. **Measured results** — Calculated actual RPO and RTO with a drill report.

### Key Takeaways

1. **Practice makes perfect** — The first time you failover should NOT be during a real disaster.
2. **Measure everything** — RTO and RPO are just numbers until you measure them.
3. **Automate what you can** — Manual failover works but is slow. Use Patroni or pg_auto_failover.
4. **Test regularly** — Run DR drills quarterly at minimum. Things change — new data, new services, new people.
5. **Document the procedure** — Can someone else do this at 3 AM on a Saturday?

### Enterprise BCDR Maturity Model

| Level | Description | This Lab |
|-------|------------|----------|
| 1 - Ad hoc | No plan, no backups | - |
| 2 - Backup | Regular backups, untested | Notebook 3 |
| 3 - Replication | Hot standby, manual failover | Notebook 2 |
| 4 - Automated | Automatic failover, monitored | Next step |
| 5 - Chaos | Regular DR drills, chaos engineering | This notebook! |

### What is Next?

To take BCDR further, explore:
- **Patroni** — Automatic PostgreSQL failover with consensus
- **Redis Sentinel** — Automatic Redis failover
- **Chaos Engineering** — Netflix Chaos Monkey, Gremlin, Litmus
- **Multi-region** — Azure Paired Regions, AWS Multi-AZ
- **Runbooks** — Documented step-by-step procedures for every failure scenario